# Implement Dot Product

**Easy** &nbsp;·&nbsp; TensorTonic &nbsp;·&nbsp; `Linear Algebra`

The dot product of two equal-length 1-D arrays `x` and `y` of length `n` is defined
algebraically as:

$$x \cdot y = \sum_{i=1}^{n} x_i y_i = x_1 y_1 + x_2 y_2 + \cdots + x_n y_n$$

and geometrically as:

$$x \cdot y = \|x\| \, \|y\| \cdot \cos(\theta)$$

where $\|x\|$ and $\|y\|$ are the magnitudes of `x` and `y`, and $\theta$ is the
angle between them. Return the dot product as a scalar float.

---

**Example 1:**

```
Input:  x = [1, 2, 3], y = [4, 5, 6]
Output: 32.0
1*4 + 2*5 + 3*6 = 4 + 10 + 18 = 32
```

**Example 2:**

```
Input:  x = [1, 0], y = [0, 1]
Output: 0.0
orthogonal vectors (perpendicular)
```

**Example 3:**

```
Input:  x = [-1, 2], y = [3, -1]
Output: -5.0
(-1)*3 + 2*(-1) = -3 + (-2) = -5
```

---

**Hint 1:** convert inputs to NumPy arrays first, then use vectorized operations.

**Hint 2:** NumPy has a built-in function for this.

**Requirements:**

- must work for lists **or** NumPy arrays
- must return a float
- must be vectorized (no Python element loops)
- must handle 1-D arrays only
- must raise `ValueError` for mismatched lengths

**Constraints:**

- time limit 200 ms, memory 64 MB
- NumPy only (no sklearn, scipy)

### Where the work actually is

The maths is one line. The requirements list is the real problem — read it again
and notice it is asking for four things the formula does not mention:

1. **"lists *or* NumPy arrays"** — so something has to normalise the input first.
   There are two functions that do this, `np.array` and `np.asarray`, and they
   differ in exactly one way. Look up what that difference is; one of them is
   free when the input is already an array.
2. **"must return a float"** — NumPy will hand you back a `np.float64`, which is
   *not* the same type. What does the test cell have to see?
3. **"1-D only"** — what attribute of an array tells you its rank?
4. **"`ValueError` for mismatched lengths"** — this one has to be *your* check.
   Decide where it goes: before or after you normalise the input?

Write the guard clauses first, then the one line of maths.

### Then, when it passes

Write a second version that does it with a Python `for` loop, and a third that
does `np.sum(x * y)`. Both are correct. The benchmark cell at the bottom will
show you what the "no Python element loops" requirement is actually protecting
you from — and whether the `np.sum` version costs anything.

In [10]:
import numpy as np

class Solution:

    def dot_product(self, x, y) -> float:
        if len(x) != len(y):
            raise ValueError("Vectors must have the same length")

        return float(np.dot(x, y))

    def dot_product_sum(self, x, y) -> float:
        if len(x) != len(y):
            raise ValueError("Vectors must have the same length")

        return float(sum(xi * yi for xi, yi in zip(x, y)))

    def dot_product_loop(self, x, y) -> float:
        if len(x) != len(y):
            raise ValueError("Vectors must have the same length")

        s: float = 0.0

        for i, j in zip(x, y):
            s += i * j

        return s

In [11]:
def check(got, want, tol=1e-9):
    """Compare one result against its expected value."""
    if got is None:
        return "not implemented"
    try:
        return "OK" if abs(got - want) < tol else f"WRONG got {got!r} want {want!r}"
    except TypeError:
        return f"WRONG got {got!r} (expected a number)"


sol = Solution()

cases = [
    ([1, 2, 3],  [4, 5, 6],  32.0),   # the worked example
    ([1, 0],     [0, 1],      0.0),   # orthogonal
    ([-1, 2],    [3, -1],    -5.0),   # negatives
    ([0, 0, 0],  [1, 2, 3],   0.0),   # zero vector
    ([2.5],      [4.0],      10.0),   # single element, floats
    (np.array([1, 2]), np.array([3, 4]), 11.0),   # ndarray input, not a list
]

for x, y, want in cases:
    print(f"{str(x):<14} . {str(y):<14} -> {check(sol.dot_product(x, y), want)}")

# requirement: a float, not a np.float64
got = sol.dot_product([1, 2], [3, 4])
print("\nreturns exactly float:", type(got).__name__ == "float", f"(got {type(got).__name__})")

# requirement: ValueError on mismatched lengths
try:
    sol.dot_product([1, 2, 3], [1, 2])
    print("mismatched lengths  -> nothing raised, expected ValueError")
except ValueError:
    print("mismatched lengths  -> ValueError  OK")

[1, 2, 3]      . [4, 5, 6]      -> OK
[1, 0]         . [0, 1]         -> OK
[-1, 2]        . [3, -1]        -> OK
[0, 0, 0]      . [1, 2, 3]      -> OK
[2.5]          . [4.0]          -> OK
[1 2]          . [3 4]          -> OK

returns exactly float: True (got float)
mismatched lengths  -> ValueError  OK


### After it passes: what the loop actually costs

Run this once all three versions work. All three are `O(n)` on paper — the
question the numbers answer is what the constant factor looks like, and whether
`np.sum(x * y)` (which builds the whole product array before adding it up) pays
for that extra memory.

In [12]:
import time

def bench(fn, x, y, repeats=5):
    best = float("inf")
    for _ in range(repeats):
        start = time.perf_counter()
        fn(x, y)
        best = min(best, time.perf_counter() - start)
    return best * 1000      # milliseconds

rng = np.random.default_rng(0)
for n in [1_000, 10_000, 100_000, 1_000_000]:
    x, y = rng.standard_normal(n), rng.standard_normal(n)
    print(f"n = {n:>9,}")
    print(f"    vectorized {bench(sol.dot_product, x, y):8.3f} ms")
    print(f"    np.sum     {bench(sol.dot_product_sum, x, y):8.3f} ms")
    print(f"    loop       {bench(sol.dot_product_loop, x.tolist(), y.tolist()):8.3f} ms")
    print()

n =     1,000
    vectorized    0.003 ms
    np.sum        0.209 ms
    loop          0.046 ms

n =    10,000
    vectorized    0.006 ms
    np.sum        2.803 ms
    loop          0.837 ms

n =   100,000
    vectorized    0.010 ms
    np.sum       28.375 ms
    loop          4.505 ms

n = 1,000,000
    vectorized    1.946 ms
    np.sum      224.894 ms
    loop         44.009 ms

